In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## Krok 1: Definujte Pydantic modely pre štruktúrované výstupy

Tieto modely definujú **schému**, ktorú agenti budú vracať. Použitie `response_format` s Pydantic zaručuje:
- ✅ Typovo bezpečné získavanie dát
- ✅ Automatickú validáciu
- ✅ Žiadne chyby parsovania z odpovedí v slobodnom texte
- ✅ Jednoduché podmienené smerovanie na základe polí


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Krok 2: Vytvorte nástroj na rezerváciu hotelov

Tento nástroj je to, čo bude volať **availability_agent** na kontrolu, či sú k dispozícii izby. Používame dekorátor `@ai_function`, aby sme:
- Prevodili funkciu v Pythone na nástroj volateľný AI
- Automaticky generovali JSON schému pre LLM
- Riešili overovanie parametrov
- Umožnili automatické vyvolanie agentmi

Pre túto ukážku:
- **Štokholm, Seattle, Tokio, Londýn, Amsterdam** → Izby sú k dispozícii ✅
- **Všetky ostatné mestá** → Izby nie sú k dispozícii ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Krok 3: Definujte funkcie podmienok pre smerovanie

Tieto funkcie kontrolujú odpoveď agenta a určujú, ktorou cestou sa bude v pracovnom postupe pokračovať.

**Kľúčový vzor:**
1. Skontrolujte, či je správa `AgentExecutorResponse`
2. Analyzujte štruktúrovaný výstup (model Pydantic)
3. Vráťte `True` alebo `False` na riadenie smerovania

Pracovný postup vyhodnotí tieto podmienky na **hranách**, aby rozhodol, ktorý vykonávateľ sa má vyvolať ďalej.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Krok 4: Vytvorenie vlastného display exekútora

Exekútory sú súčasti pracovného toku, ktoré vykonávajú transformácie alebo vedľajšie efekty. Používame dekorátor `@executor` na vytvorenie vlastného exekútora, ktorý zobrazuje konečný výsledok.

**Kľúčové koncepty:**
- `@executor(id="...")` - Registruje funkciu ako exekútor pracovného toku
- `WorkflowContext[Never, str]` - Typové nápovedy pre vstup/výstup
- `ctx.yield_output(...)` - Vracia konečný výsledok pracovného toku


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Krok 5: Načítať premenné prostredia

Nakonfigurujte klienta LLM. Tento príklad funguje s:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — odporúčané)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## Krok 6: Vytvorenie AI agentov so štruktúrovaným výstupom

Vytvárame **tri špecializované agentov**, z ktorých každý je zabalený v `AgentExecutor`:

1. **availability_agent** - Kontroluje dostupnosť hotelov pomocou nástroja
2. **alternative_agent** - Navrhuje alternatívne mestá (keď nie sú žiadne izby)
3. **booking_agent** - Povzbudzuje k rezervácii (keď sú izby dostupné)

**Hlavné vlastnosti:**
- `tools=[hotel_booking]` - Poskytuje nástroj agentovi
- `response_format=PydanticModel` - Vynucuje štruktúrovaný JSON výstup
- `AgentExecutor(..., id="...")` - Zabaluje agenta pre použitie v pracovnom postupe


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## Krok 7: Vytvorenie Workflow s podmienenými hranami

Teraz použijeme `WorkflowBuilder` na zostavenie grafu s podmieneným smerovaním:

**Štruktúra workflow:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Kľúčové metódy:**
- `.set_start_executor(...)` - Nastaví východiskový bod
- `.add_edge(from, to, condition=...)` - Pridá podmienenú hranu
- `.build()` - Finalizuje workflow


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Krok 8: Spustenie testovacieho prípadu 1 - Mesto BEZ dostupnosti (Paríž)

Otestujeme cestu **bez dostupnosti** požiadaním o hotely v Paríži (ktorý v našej simulácii nemá žiadne izby).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Krok 9: Spustiť testovací prípad 2 - Mesto S dostupnosťou (Štokholm)

Teraz otestujme cestu **dostupnosti** žiadosťou o hotely v Štokholme (ktorý v našej simulácii má voľné izby).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Kľúčové poznatky a ďalšie kroky

### ✅ Čo ste sa naučili:

1. **Vzorec WorkflowBuilder**
   - Použite `.set_start_executor()` na definovanie vstupného bodu
   - Použite `.add_edge(from, to, condition=...)` pre podmienené smerovanie
   - Zavolajte `.build()` na dokončenie pracovného postupu

2. **Podmienené smerovanie**
   - Funkcie podmienok kontrolujú `AgentExecutorResponse`
   - Parsujú štruktúrované výstupy na prijímanie rozhodnutí o smerovaní
   - Vráťte `True` na aktivovanie hrany, `False` na jej preskočenie

3. **Integrácia nástrojov**
   - Použite `@ai_function` na premenenie Python funkcií na AI nástroje
   - Agent automaticky volá nástroje podľa potreby
   - Nástroje vracajú JSON, ktorý agenti vedia spracovať

4. **Štruktúrované výstupy**
   - Používajte Pydantic modely na typovo bezpečný extrahovanie dát
   - Nastavte `response_format=MyModel` pri vytváraní agentov
   - Odozvy parsujte pomocou `Model.model_validate_json()`

5. **Vlastní exekútori**
   - Použite `@executor(id="...")` na vytvorenie komponentov workflow
   - Exekútori môžu transformovať dáta alebo vykonávať vedľajšie účinky
   - Použite `ctx.yield_output()` na produkciu výsledkov workflow

### 🚀 Aplikácie v reálnom svete:

- **Rezervácie cestovania**: Kontrola dostupnosti, navrhovanie alternatív, porovnávanie možností
- **Zákaznícka podpora**: Smerovanie podľa typu problému, sentimentu, priority
- **E-commerce**: Kontrola zásob, navrhovanie alternatív, spracovanie objednávok
- **Moderovanie obsahu**: Smerovanie podľa skóre toxicity, používateľských hlásení
- **Schvaľovacie workflow**: Smerovanie podľa sumy, používateľskej roly, úrovne rizika
- **Viacfázové spracovanie**: Smerovanie podľa kvality dát, úplnosti

### 📚 Ďalšie kroky:

- Pridať komplexnejšie podmienky (viac kritérií)
- Implementovať slučky so správou stavu workflow
- Pridať pod-workflow pre znovupoužiteľné komponenty
- Integrovať sa s reálnymi API (rezervácie hotelov, systémy zásob)
- Pridať spracovanie chýb a záložné cesty
- Vizualizovať workflow pomocou vstavaných nástrojov na vizualizáciu


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
